# Incompressible Navier-Stokes with Passive Temperature Scalar

## Overview

This notebook documents [`src/incomp_w_temp.jl`](src/incomp_w_temp.jl) — a finite element simulation of **2D incompressible Navier-Stokes flow with a passively advected temperature scalar** around a circular obstacle (the classical **von Kármán vortex street** benchmark).

The solver is implemented using:
- **[Ferrite.jl](https://ferrite-fem.github.io/)** — finite element assembly and degree-of-freedom management
- **[FerriteGmsh.jl](https://github.com/Ferrite-FEM/FerriteGmsh.jl)** — mesh generation via Gmsh
- **[OrdinaryDiffEq.jl](https://docs.sciml.ai/OrdinaryDiffEq/stable/)** — implicit time integration (Rosenbrock `Rodas5P`)
- **[WriteVTK.jl](https://github.com/jipolanco/WriteVTK.jl)** — output for Paraview visualization

---

## Table of Contents
1. [Governing Equations](#governing-equations)
2. [Import Required Libraries](#imports)
3. [Physical Parameters](#parameters)
4. [Mesh Generation with Gmsh](#mesh)
5. [Finite Element Spaces](#fe-spaces)
6. [Boundary Conditions](#bcs)
7. [Weak Formulation](#weak-form)
8. [Mass Matrix Assembly](#mass)
9. [Stiffness Matrix Assembly](#stiffness)
10. [Nonlinear Right-Hand Side & Jacobian](#nonlinear)
11. [ODE Problem Setup & Time Integration](#ode)
12. [VTK Output](#vtk)

---
## 1. Governing Equations <a name="governing-equations"></a>

### 1.1 Incompressible Navier-Stokes Equations

We seek the velocity field $\mathbf{v}(\mathbf{x},t) \in \mathbb{R}^2$ and pressure $p(\mathbf{x},t) \in \mathbb{R}$ satisfying:

$$\frac{\partial \mathbf{v}}{\partial t} + (\mathbf{v} \cdot \nabla)\mathbf{v} = -\nabla p + \nu \,\Delta \mathbf{v} \quad \text{in } \Omega \times (0, T]$$

$$\nabla \cdot \mathbf{v} = 0 \quad \text{in } \Omega \times (0, T]$$

where:
- $\Omega \subset \mathbb{R}^2$ is the spatial domain (rectangle minus cylinder)
- $\nu > 0$ is the **kinematic viscosity** (here $\nu = 10^{-3}$, giving $Re \approx 100$)
- The nonlinear term $(\mathbf{v} \cdot \nabla)\mathbf{v}$ represents inertial convection

### 1.2 Passive Temperature Transport (Advection-Diffusion)

Temperature $T(\mathbf{x}, t)$ is transported by the flow as a **passive scalar** — it does not feed back into the momentum equation:

$$\frac{\partial T}{\partial t} + \mathbf{v} \cdot \nabla T = \kappa \, \Delta T \quad \text{in } \Omega \times (0, T]$$

where $\kappa > 0$ is the **thermal diffusivity** (here $\kappa = 10^{-4}$, giving $Pe = \nu/\kappa = 10$).

### 1.3 Coupled System

Writing the state vector $U = [\mathbf{v}, p, T]^\top$, the full system is:

$$\begin{pmatrix} \mathbf{I} & 0 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & 1 \end{pmatrix} \dot{U} = \underbrace{\begin{pmatrix} \nu\Delta\mathbf{v} - \nabla p \\ -\nabla\cdot\mathbf{v} \\ \kappa\Delta T \end{pmatrix}}_{\text{linear: }KU} + \underbrace{\begin{pmatrix} -(\mathbf{v}\cdot\nabla)\mathbf{v} \\ 0 \\ -\mathbf{v}\cdot\nabla T \end{pmatrix}}_{\text{nonlinear: }N(U)}$$

> **Note:** The pressure row has no time derivative — it acts as a Lagrange multiplier enforcing incompressibility. The resulting mass matrix $M$ is **singular** (DAE structure), handled by the implicit solver.

### 1.4 Boundary & Initial Conditions

| Boundary | $\mathbf{v}$ | $T$ |
|---|---|---|
| Left (inlet) | Parabolic: $v_x = \frac{4 v_{in}(t)\, y (H-y)}{H^2}$, $v_y=0$ | $T = T_{in}(t)$ |
| Top & Bottom (walls) | No-slip: $\mathbf{v} = \mathbf{0}$ | — (free) |
| Cylinder (hole) | No-slip: $\mathbf{v} = \mathbf{0}$ | — (free) |
| Right (outflow) | Free (do nothing) | Free |

Both inlet profiles ramp linearly from $0$ to their maximum over $t \in [0, 1]$:

$$v_{in}(t) = \min(1.5\, t,\ 1.5), \qquad T_{in}(t) = \min(1.5\, t,\ 1.5)$$

The initial condition is $U(\mathbf{x}, 0) = \mathbf{0}$.

---
## 2. Import Required Libraries <a name="imports"></a>

| Package | Role |
|---|---|
| `Ferrite` | Core FEM: mesh, DOFs, assembly, BCs |
| `FerriteGmsh` / `Gmsh` | Mesh generation via the Gmsh API |
| `SparseArrays` | Sparse matrix storage for $M$, $K$, $J$ |
| `BlockArrays` | Block structure for element matrices $M_e$, $K_e$ |
| `LinearAlgebra` | BLAS operations, `mul!` |
| `UnPack` | `@unpack` macro for struct fields |
| `LinearSolve` | Linear solver backend used internally by the ODE solver |
| `WriteVTK` | VTK file output for Paraview |
| `OrdinaryDiffEq` | `Rodas5P` stiff implicit time integrator |
| `DiffEqBase` | Common interface: `ODEFunction`, `ODEProblem` |

In [ ]:
println("Starting code execution")

using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, UnPack, LinearSolve, WriteVTK
using OrdinaryDiffEq
using DiffEqBase
using FerriteGmsh
using FerriteGmsh: Gmsh

println("Dependencies successfully loaded")

---
## 3. Physical Parameters <a name="parameters"></a>

The two dimensionless-like parameters govern the balance between convection and diffusion:

$$\nu = \frac{1}{1000} = 10^{-3} \qquad \text{(kinematic viscosity [m}^2\text{/s])}$$

$$\kappa = 10^{-4} \qquad \text{(thermal diffusivity [m}^2\text{/s])}$$

The relevant dimensionless numbers for this benchmark are:

$$Re = \frac{U_\text{max}\, D}{\nu} = \frac{1.5 \times 0.1}{10^{-3}} = 150 \quad \text{(Reynolds number)}$$

$$Pe = \frac{U_\text{max}\, D}{\kappa} = \frac{1.5 \times 0.1}{10^{-4}} = 1500 \quad \text{(Péclet number)}$$

where $D = 0.1$ m is the cylinder diameter and $U_\text{max} = 1.5$ m/s is the peak inflow velocity.
At $Re = 150$, the flow is expected to exhibit the classic **von Kármán vortex street** (periodic vortex shedding).

In [ ]:
# Kinematic viscosity — controls diffusion of momentum
ν = 1.0 / 1000.0  # ν = 1e-3 m²/s

# Thermal diffusivity — controls diffusion of temperature
κ = 1.0e-4        # κ = 1e-4 m²/s

# Derived dimensionless numbers (informational)
D = 0.1        # cylinder diameter [m]
U_max = 1.5    # peak inflow velocity [m/s]
Re = U_max * D / ν   # Reynolds number
Pe = U_max * D / κ   # Péclet number
println("Reynolds number Re = $Re")
println("Péclet number   Pe = $Pe")

---
## 4. Mesh Generation with Gmsh <a name="mesh"></a>

### Domain Geometry

The domain $\Omega$ is a **rectangle with a circular hole**:

$$\Omega = [0,\ 1.1] \times [0,\ 0.41] \setminus \overline{B}(\mathbf{x}_c,\, r)$$

where:
- Rectangle dimensions: $1.1 \,\text{m} \times 0.41 \,\text{m}$
- Circle center: $\mathbf{x}_c = (0.2,\, 0.2)$
- Circle radius: $r = 0.05$ m (diameter $D = 0.1$ m, slightly off-center to trigger vortex shedding)

```
     ┌─────────────────────────────────────────┐
     │          top wall                        │
     │   ┌─────────────────────────────────    │←right
left→│   ○ (cylinder)       channel flow ...   │
     │   └─────────────────────────────────    │
     │          bottom wall                     │
     └─────────────────────────────────────────┘
```

### Physical Groups (Boundary Labels)

| Tag | Name | Role |
|---|---|---|
| 6 | `bottom` | No-slip wall |
| 7 | `left` | Inflow |
| 8 | `right` | Outflow (free) |
| 9 | `top` | No-slip wall |
| 5 | `hole` | Cylinder surface (no-slip) |

### Mesh Settings
- **Algorithm 11** (Delaunay-type for quads via transfinite)
- `MeshSizeFromCurvature = 20` — automatic refinement near the curved cylinder boundary
- `MeshSizeMax = 0.05` m — global maximum element size

The mesh is generated and converted to a Ferrite `Grid` via `togrid()`.

In [ ]:
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)
dim = 2

# --- Geometry ---
# Create bounding rectangle [0,1.1] x [0,0.41]
rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)

# Create circular obstacle: center (0.2, 0.2), radius 0.05
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])

# Boolean cut: Ω = rectangle \ circle
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])
gmsh.model.occ.synchronize()

# --- Physical groups (boundary labels) ---
bottomtag = gmsh.model.model.add_physical_group(dim - 1, [6], -1, "bottom")
lefttag   = gmsh.model.model.add_physical_group(dim - 1, [7], -1, "left")
righttag  = gmsh.model.model.add_physical_group(dim - 1, [8], -1, "right")
toptag    = gmsh.model.model.add_physical_group(dim - 1, [9], -1, "top")
holetag   = gmsh.model.model.add_physical_group(dim - 1, [5], -1, "hole")

# --- Mesh options ---
gmsh.option.setNumber("Mesh.Algorithm", 11)              # Packing of parallelograms
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)  # Refine near cylinder
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)          # Global max element size ≈ 5cm

# Generate and import into Ferrite
gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize()

println("Meshing done — $(getncells(grid)) cells, $(getnnodes(grid)) nodes")

---
## 5. Finite Element Spaces <a name="fe-spaces"></a>

### Element Choices

We use a **mixed finite element** formulation on quadrilateral cells:

| Field | Symbol | Space | Interpolation | Motivation |
|---|---|---|---|---|
| Velocity | $\mathbf{v}$ | $[Q_2]^2$ | Biquadratic Lagrange, vectorial | High accuracy for velocity |
| Pressure | $p$ | $Q_1$ | Bilinear Lagrange | LBB-stable with $Q_2$ velocity |
| Temperature | $T$ | $Q_1$ | Bilinear Lagrange | Sufficient for smooth scalar field |

This **Taylor-Hood-like Q2/Q1 pair** satisfies the inf-sup (LBB) stability condition, preventing spurious pressure modes.

### Quadrature

A **4-point Gauss quadrature** rule on `RefQuadrilateral` is used, which exactly integrates polynomials up to degree 7 — sufficient for the $Q_2$ basis functions ($2 \times 2$ tensor products).

### DOF Layout

The `DofHandler` organizes DOFs as a concatenated global vector:
$$U_h = [\underbrace{v_1, v_2, \ldots}_{\text{velocity DOFs}},\ \underbrace{p_1, \ldots}_{\text{pressure DOFs}},\ \underbrace{T_1, \ldots}_{\text{temperature DOFs}}]^\top$$

This block structure is exploited in the mass and stiffness matrix assembly.

In [ ]:
# --- Interpolation spaces ---
ip_v = Lagrange{RefQuadrilateral, 2}()^2  # Q2 vector: biquadratic velocity
ip_p = Lagrange{RefQuadrilateral, 1}()     # Q1 scalar: bilinear pressure
ip_T = Lagrange{RefQuadrilateral, 1}()     # Q1 scalar: bilinear temperature

# --- Quadrature rule ---
# 4-point Gauss rule; integrates polynomials up to degree 7 exactly
qr = QuadratureRule{RefQuadrilateral}(4)

# --- Cell values: shape functions + gradients at quadrature points ---
cellvalues_v = CellValues(qr, ip_v)   # for velocity
cellvalues_p = CellValues(qr, ip_p)   # for pressure
cellvalues_T = CellValues(qr, ip_T)   # for temperature

# --- Degree-of-freedom handler ---
dh = DofHandler(grid)
add!(dh, :v, ip_v)   # velocity field
add!(dh, :p, ip_p)   # pressure field
add!(dh, :T, ip_T)   # temperature field
close!(dh)

println("Total DOFs: $(ndofs(dh))")
println("  Velocity DOFs : $(length(dof_range(dh, :v)))")
println("  Pressure DOFs : $(length(dof_range(dh, :p)))")
println("  Temperature DOFs: $(length(dof_range(dh, :T)))")

---
## 6. Boundary Conditions <a name="bcs"></a>

Dirichlet boundary conditions are enforced strongly via Ferrite's `ConstraintHandler`.

### No-slip Velocity (Walls + Cylinder)

On $\partial\Omega_\text{noslip} = \text{top} \cup \text{bottom} \cup \text{hole}$:

$$\mathbf{v} = \mathbf{0} \quad \text{on } \partial\Omega_\text{noslip}$$

This enforces the physical no-slip condition: fluid velocity matches the (stationary) wall.

### Parabolic Inflow Velocity

On $\partial\Omega_\text{inlet}$ (left boundary), a **Poiseuille-type profile** is imposed:

$$v_x(y, t) = \frac{4\, v_{in}(t) \cdot y\, (H - y)}{H^2}, \quad v_y = 0$$

where $H = 0.41$ m is the channel height and $v_{in}(t) = \min(1.5\,t,\, 1.5)$.

At $t \geq 1$ s, the steady-state maximum is $v_x^\text{max} = 1.5$ m/s at $y = H/2$.

The average (bulk) velocity is:

$$\bar{v}_x = \frac{2}{3} v_{in}(t) = 1.0 \text{ m/s at steady state}$$

### Inlet Temperature

On $\partial\Omega_\text{inlet}$, the temperature ramps identically:

$$T(y, t) = T_{in}(t) = \min(1.5\,t,\, 1.5)$$

> Note: No temperature BC is imposed on walls or the cylinder — this corresponds to homogeneous Neumann (zero heat flux, adiabatic walls).

### Constraint Handler

`ConstraintHandler(dh)` manages all constraints and provides:
- `apply!(u, ch)` — project DOFs onto constraint manifold
- `apply!(K, ch)` — modify stiffness matrix for Dirichlet BCs
- `update!(ch, t)` — update time-dependent BC values

In [ ]:
ch = ConstraintHandler(dh)

# --- No-slip: v = 0 on top, bottom, cylinder ---
nosplip_facet_names = ["top", "bottom", "hole"]
∂Ω_noslip = union(getfacetset.((grid,), nosplip_facet_names)...)
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc)

# --- Parabolic inflow: v_x = 4*v_in(t)*y*(H-y)/H², v_y = 0 ---
∂Ω_inflow = getfacetset(grid, "left")
H = 0.41                                  # channel height [m]
vᵢₙ(t) = min(t * 1.5, 1.5)              # ramp: 0 → 1.5 over t ∈ [0,1]
parabolic_inflow_profile(x, t) = Vec((4 * vᵢₙ(t) * x[2] * (H - x[2]) / H^2, 0.0))
inflow_bc = Dirichlet(:v, ∂Ω_inflow, parabolic_inflow_profile, [1, 2])
add!(ch, inflow_bc)

# --- Inlet temperature: T = T_in(t); adiabatic elsewhere ---
Tᵢₙ(t) = min(t * 1.5, 1.5)             # same ramp profile
T_inlet(x, t) = Tᵢₙ(t)
temp_bc = Dirichlet(:T, ∂Ω_inflow, (x, t) -> T_inlet(x, t), [1])
add!(ch, temp_bc)

close!(ch)
update!(ch, 0.0)   # initialize BCs at t = 0

println("Constraints applied: $(length(ch.prescribed_dofs)) prescribed DOFs")

---
## 7. Weak Formulation <a name="weak-form"></a>

### Test Function Spaces

Let $\mathbf{w} \in [H^1_0(\Omega)]^2$, $q \in L^2(\Omega)$, $s \in H^1(\Omega)$ be test functions for velocity, pressure, and temperature respectively (with appropriate homogeneous Dirichlet conditions built in).

### Momentum Equation (Velocity)

Multiply by $\mathbf{w}$ and integrate by parts:

$$\int_\Omega \frac{\partial \mathbf{v}}{\partial t} \cdot \mathbf{w}\, d\Omega - \int_\Omega p\, (\nabla \cdot \mathbf{w})\, d\Omega + \int_\Omega \nu\, \nabla\mathbf{v} : \nabla\mathbf{w}\, d\Omega + \int_\Omega (\mathbf{v} \cdot \nabla)\mathbf{v} \cdot \mathbf{w}\, d\Omega = 0$$

### Continuity Equation (Pressure)

$$\int_\Omega q\, (\nabla \cdot \mathbf{v})\, d\Omega = 0$$

### Temperature Equation

Multiply by $s$, integrate by parts:

$$\int_\Omega \frac{\partial T}{\partial t} s\, d\Omega + \int_\Omega \kappa\, \nabla T \cdot \nabla s\, d\Omega + \int_\Omega (\mathbf{v} \cdot \nabla T)\, s\, d\Omega = 0$$

### Matrix Form After Discretization

After FE discretization with basis functions $\{\boldsymbol{\varphi}_i\}$ for $\mathbf{v}$, $\{\psi_j\}$ for $p$, and $\{\phi_k\}$ for $T$, the system becomes:

$$M \dot{U} = K U + N(U)$$

where the **global mass matrix** is:

$$M = \begin{pmatrix} M_{vv} & 0 & 0 \\ 0 & 0 & 0 \\ 0 & 0 & M_{TT} \end{pmatrix}, \quad (M_{vv})_{ij} = \int_\Omega \boldsymbol{\varphi}_i \cdot \boldsymbol{\varphi}_j\, d\Omega, \quad (M_{TT})_{kl} = \int_\Omega \phi_k \phi_l\, d\Omega$$

The **global stiffness matrix** $K$ is:

$$K = \begin{pmatrix} K_{vv} & K_{vp} & 0 \\ K_{pv} & 0 & 0 \\ 0 & 0 & K_{TT} \end{pmatrix}$$

with entries:

$$(K_{vv})_{ij} = -\int_\Omega \nu\, \nabla\boldsymbol{\varphi}_i : \nabla\boldsymbol{\varphi}_j\, d\Omega$$

$$(K_{vp})_{ij} = \int_\Omega (\nabla \cdot \boldsymbol{\varphi}_i)\, \psi_j\, d\Omega, \qquad K_{pv} = K_{vp}^\top$$

$$(K_{TT})_{kl} = -\int_\Omega \kappa\, \nabla\phi_k \cdot \nabla\phi_l\, d\Omega$$

The **nonlinear vector** $N(U) = [N_v(U),\ 0,\ N_T(U)]^\top$:

$$(N_v)_j = -\int_\Omega (\mathbf{v} \cdot \nabla\mathbf{v}) \cdot \boldsymbol{\varphi}_j\, d\Omega, \qquad (N_T)_k = -\int_\Omega (\mathbf{v} \cdot \nabla T)\, \phi_k\, d\Omega$$

---
## 8. Mass Matrix Assembly <a name="mass"></a>

`assemble_mass_matrix` loops over all cells and accumulates element-level contributions into the global sparse $M$.

### Element Mass Matrix Structure

The element matrix $M_e$ is a blocked $[n_v + n_p + n_T] \times [n_v + n_p + n_T]$ matrix:

$$M_e = \begin{pmatrix} M_e^{vv} & 0 & 0 \\ 0 & \mathbf{0} & 0 \\ 0 & 0 & M_e^{TT} \end{pmatrix}$$

**Velocity block** (i,j both index velocity basis):

$$(M_e^{vv})_{ij} = \int_{\Omega_e} \boldsymbol{\varphi}_i \cdot \boldsymbol{\varphi}_j\, d\Omega \approx \sum_{q} \boldsymbol{\varphi}_i(\mathbf{x}_q) \cdot \boldsymbol{\varphi}_j(\mathbf{x}_q)\, w_q\, |\text{det}\,J_q|$$

**Pressure block** is **zero** — pressure has no time derivative (constraint equation).

**Temperature block** (i,j both index temperature basis):

$$(M_e^{TT})_{ij} = \int_{\Omega_e} \phi_i \phi_j\, d\Omega \approx \sum_{q} \phi_i(\mathbf{x}_q)\, \phi_j(\mathbf{x}_q)\, w_q\, |\text{det}\,J_q|$$

### After Assembly

`apply!(M, ch)` modifies rows/columns of $M$ corresponding to constrained DOFs so that the ODE solver handles the DAE structure correctly.

In [ ]:
"""
    assemble_mass_matrix(cellvalues_v, cellvalues_p, cellvalues_T, M, dh)

Assemble the global mass matrix M for the (v, p, T) system.
Only velocity (block 1,1) and temperature (block 3,3) have entries.
Pressure has no time derivative, so its block is zero (DAE structure).
"""
function assemble_mass_matrix(cellvalues_v, cellvalues_p, cellvalues_T, M::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs_T = getnbasefunctions(cellvalues_T)
    n_basefuncs = n_basefuncs_v + n_basefuncs_p + n_basefuncs_T

    v▄, p▄, T▄ = 1, 2, 3
    # Element matrix with block structure [v|p|T] × [v|p|T]
    Mₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p, n_basefuncs_T],
                      [n_basefuncs_v, n_basefuncs_p, n_basefuncs_T])

    mass_assembler = start_assemble(M)
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_T, cell)

        # Block (1,1): Mᵉᵥᵥ = ∫ φᵢ · φⱼ dΩ
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += φᵢ ⋅ φⱼ * dΩ
                end
            end
        end

        # Block (3,3): MᵉTT = ∫ φᵢ φⱼ dΩ  (note: scalar, not dot product)
        for q_point in 1:getnquadpoints(cellvalues_T)
            dΩ = getdetJdV(cellvalues_T, q_point)
            for i in 1:n_basefuncs_T
                φᵢ = shape_value(cellvalues_T, q_point, i)
                for j in 1:n_basefuncs_T
                    φⱼ = shape_value(cellvalues_T, q_point, j)
                    Mₑ[BlockIndex((T▄, T▄), (i, j))] += φᵢ * φⱼ * dΩ
                end
            end
        end
        # Block (2,2) = 0 — pressure has no time derivative

        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    return M
end

M = allocate_matrix(dh)
M = assemble_mass_matrix(cellvalues_v, cellvalues_p, cellvalues_T, M, dh)
println("Mass matrix assembled: $(size(M)) sparse matrix, $(nnz(M)) nonzeros")

---
## 9. Stiffness Matrix Assembly <a name="stiffness"></a>

`assemble_stiffness_matrix` assembles all **linear** spatial operators into $K$.

### Element Stiffness Matrix Structure

$$K_e = \begin{pmatrix} K_e^{vv} & K_e^{vp} & 0 \\ K_e^{pv} & 0 & 0 \\ 0 & 0 & K_e^{TT} \end{pmatrix}$$

**Viscous velocity block** — from integration by parts of $\nu\Delta\mathbf{v}$:

$$(K_e^{vv})_{ij} = -\int_{\Omega_e} \nu\, \nabla\boldsymbol{\varphi}_i : \nabla\boldsymbol{\varphi}_j\, d\Omega$$

Here $\nabla\boldsymbol{\varphi}_i : \nabla\boldsymbol{\varphi}_j = \sum_{\alpha,\beta} \partial_\alpha (\varphi_i)_\beta \cdot \partial_\alpha (\varphi_j)_\beta$ (double contraction `⊡`).

**Velocity-pressure coupling blocks** — from $-\nabla p$ and $\nabla \cdot \mathbf{v} = 0$:

$$(K_e^{vp})_{ij} = \int_{\Omega_e} (\nabla \cdot \boldsymbol{\varphi}_i)\, \psi_j\, d\Omega$$
$$(K_e^{pv})_{ji} = (K_e^{vp})_{ij}$$

This saddle-point structure ensures incompressibility is a constraint.

**Thermal diffusion block** — from integration by parts of $\kappa\Delta T$:

$$(K_e^{TT})_{ij} = -\int_{\Omega_e} \kappa\, \nabla\phi_i \cdot \nabla\phi_j\, d\Omega$$

In [ ]:
"""
    assemble_stiffness_matrix(cellvalues_v, cellvalues_p, cellvalues_T, ν, κ, K, dh)

Assemble the global stiffness matrix K containing:
  - Viscous diffusion for velocity:   K[v,v] = -ν ∫ ∇φᵢ:∇φⱼ dΩ
  - Pressure-velocity coupling:       K[v,p] = ∫ (∇·φᵢ)ψⱼ dΩ  (and transpose)
  - Thermal diffusion for temperature: K[T,T] = -κ ∫ ∇φᵢ·∇φⱼ dΩ
All integrals are approximated by Gauss quadrature.
"""
function assemble_stiffness_matrix(cellvalues_v, cellvalues_p, cellvalues_T, ν, κ, K::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs_T = getnbasefunctions(cellvalues_T)
    n_basefuncs = n_basefuncs_v + n_basefuncs_p + n_basefuncs_T
    v▄, p▄, T▄ = 1, 2, 3
    Kₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                      [n_basefuncs_v, n_basefuncs_p, n_basefuncs_T],
                      [n_basefuncs_v, n_basefuncs_p, n_basefuncs_T])

    stiffness_assembler = start_assemble(K)
    for cell in CellIterator(dh)
        fill!(Kₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)
        Ferrite.reinit!(cellvalues_T, cell)

        # Block (1,1): Kᵉᵥᵥ = -ν ∫ ∇φᵢ ⊡ ∇φⱼ dΩ
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    ∇φⱼ = shape_gradient(cellvalues_v, q_point, j)
                    Kₑ[BlockIndex((v▄, v▄), (i, j))] -= ν * ∇φᵢ ⊡ ∇φⱼ * dΩ
                end
            end
        end

        # Blocks (1,2) and (2,1): Kᵉᵥₚ = ∫ (∇·φᵢ) ψⱼ dΩ  —  saddle-point structure
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for j in 1:n_basefuncs_p
                ψ = shape_value(cellvalues_p, q_point, j)
                for i in 1:n_basefuncs_v
                    divφ = shape_divergence(cellvalues_v, q_point, i)
                    Kₑ[BlockIndex((v▄, p▄), (i, j))] += (divφ * ψ) * dΩ   # velocity eq ← pressure
                    Kₑ[BlockIndex((p▄, v▄), (j, i))] += (ψ * divφ) * dΩ   # continuity eq
                end
            end
        end

        # Block (3,3): KᵉTT = -κ ∫ ∇φᵢ · ∇φⱼ dΩ
        for q_point in 1:getnquadpoints(cellvalues_T)
            dΩ = getdetJdV(cellvalues_T, q_point)
            for i in 1:n_basefuncs_T
                ∇φᵢ = shape_gradient(cellvalues_T, q_point, i)
                for j in 1:n_basefuncs_T
                    ∇φⱼ = shape_gradient(cellvalues_T, q_point, j)
                    Kₑ[BlockIndex((T▄, T▄), (i, j))] -= κ * (∇φᵢ ⋅ ∇φⱼ) * dΩ
                end
            end
        end

        assemble!(stiffness_assembler, celldofs(cell), Kₑ)
    end
    return K
end

K = allocate_matrix(dh)
K = assemble_stiffness_matrix(cellvalues_v, cellvalues_p, cellvalues_T, ν, κ, K, dh)
println("Stiffness matrix assembled: $(size(K)) sparse matrix, $(nnz(K)) nonzeros")

# Initial condition: start from rest, then project BCs
u₀ = zeros(ndofs(dh))
apply!(u₀, ch)

# Jacobian sparsity pattern = sparsity of K
jac_sparsity = sparse(K)

# Apply constraint modifications to mass matrix
apply!(M, ch)

---
## 10. Nonlinear Right-Hand Side & Jacobian <a name="nonlinear"></a>

### 10.1 Overall RHS Structure

The full RHS is $R(U) = KU + N(U)$. The ODE solver receives `navierstokes_temp!` which computes $R(U)$.

### 10.2 Velocity Convection — Element RHS

The nonlinear momentum convection contributes to the velocity part of $N(U)$:

$$(N_v)_j = -\int_\Omega (\mathbf{v} \cdot \nabla\mathbf{v}) \cdot \boldsymbol{\varphi}_j\, d\Omega$$

Element-level (function `navierstokes_rhs_element!`):

$$(dv_e)_j \mathrel{-}= \sum_q \left(\mathbf{v}(\mathbf{x}_q) \cdot \nabla\mathbf{v}(\mathbf{x}_q)^\top\right) \cdot \boldsymbol{\varphi}_j(\mathbf{x}_q)\; w_q\, |\det J_q|$$

where $\mathbf{v}$ and $\nabla\mathbf{v}$ are reconstructed from local DOFs via:
$$\mathbf{v}(\mathbf{x}_q) = \sum_i v_i\, \boldsymbol{\varphi}_i(\mathbf{x}_q), \qquad \nabla\mathbf{v}(\mathbf{x}_q) = \sum_i v_i \otimes \nabla\boldsymbol{\varphi}_i(\mathbf{x}_q)$$

### 10.3 Temperature Advection — Element RHS

$$(N_T)_k = -\int_\Omega (\mathbf{v} \cdot \nabla T)\, \phi_k\, d\Omega$$

Element-level (function `temp_advection_element!`):

$$(dT_e)_k \mathrel{-}= \sum_q \left(\mathbf{v}(\mathbf{x}_q) \cdot \nabla T(\mathbf{x}_q)\right) \phi_k(\mathbf{x}_q)\; w_q\, |\det J_q|$$

### 10.4 Jacobian Structure

The Jacobian $J = \partial R / \partial U = K + C(U)$ where $C(U)$ is the linearization of $N(U)$:

$$C(U) = \begin{pmatrix} C_{vv}(U) & 0 & 0 \\ 0 & 0 & 0 \\ C_{Tv}(U) & 0 & C_{TT}(U) \end{pmatrix}$$

**Velocity-velocity block** (linearize $-(\mathbf{v}\cdot\nabla)\mathbf{v}$ w.r.t. $\mathbf{v}$, product rule):

$$(C_{vv})_{ji} = -\int_\Omega \left(\boldsymbol{\varphi}_i \cdot \nabla\mathbf{v}^\top + \mathbf{v} \cdot \nabla\boldsymbol{\varphi}_i^\top\right) \cdot \boldsymbol{\varphi}_j\, d\Omega$$

**Temperature-temperature block** (linearize $-\mathbf{v}\cdot\nabla T$ w.r.t. $T$):

$$(C_{TT})_{ij} = -\int_\Omega \phi_i\, (\mathbf{v} \cdot \nabla\phi_j)\, d\Omega$$

**Temperature-velocity block** (linearize $-\mathbf{v}\cdot\nabla T$ w.r.t. $\mathbf{v}$, off-diagonal coupling):

$$(C_{Tv})_{ki} = -\int_\Omega (\boldsymbol{\varphi}_i \cdot \nabla T)\, \phi_k\, d\Omega$$

> This off-diagonal block makes the Jacobian **non-symmetric** and couples the temperature equation to velocity perturbations.

In [ ]:
# ---- Parameter struct passed to ODE solver ----
struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler
    cellvalues_v::CellValues
    cellvalues_T::CellValues
    u::Vector   # scratch space for constraint-applied solution
end
p = RHSparams(K, ch, dh, cellvalues_v, cellvalues_T, copy(u₀))

# Step limiter: enforce Dirichlet BCs after each accepted step
function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

# ---- Element-level velocity convection RHS ----
# Computes:  dvₑ[j] -= (v · ∇v') · φⱼ  dΩ  at each quadrature point
function navierstokes_rhs_element!(dvₑ, vₑ, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)   # 2×2 gradient tensor
        v  = function_value(cellvalues_v, q_point, vₑ)       # velocity vector at q_point
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            dvₑ[j] -= v ⋅ ∇v' ⋅ φⱼ * dΩ   # note: v·(∇v)ᵀ·φⱼ
        end
    end
    return
end

# ---- Element-level temperature advection RHS ----
# Computes:  dTₑ[j] -= (v · ∇T) φⱼ  dΩ
function temp_advection_element!(dTₑ, Tₑ, vₑ, cellvalues_T, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_T)
    for q_point in 1:getnquadpoints(cellvalues_T)
        dΩ = getdetJdV(cellvalues_T, q_point)
        ∇T = function_gradient(cellvalues_T, q_point, Tₑ)   # temperature gradient
        v  = function_value(cellvalues_v, q_point, vₑ)       # advecting velocity
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_T, q_point, j)
            dTₑ[j] -= (v ⋅ ∇T) * φⱼ * dΩ
        end
    end
    return
end

# ---- Global RHS: du = K*u + N(u) ----
function navierstokes_temp!(du, u_uc, p::RHSparams, t)
    @unpack K, ch, dh, cellvalues_v, cellvalues_T, u = p
    u .= u_uc
    update!(ch, t); apply!(u, ch)

    # Linear part: du ← K u
    mul!(du, K, u)

    # Nonlinear velocity convection
    v_range = dof_range(dh, :v)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    vₑ = zeros(n_basefuncs_v); duₑ = zeros(n_basefuncs_v)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        vₑ .= @views u[v_celldofs]
        fill!(duₑ, 0.0)
        navierstokes_rhs_element!(duₑ, vₑ, cellvalues_v)
        assemble!(du, v_celldofs, duₑ)
    end

    # Nonlinear temperature advection
    T_range = dof_range(dh, :T)
    n_basefuncs_T = getnbasefunctions(cellvalues_T)
    Tₑ = zeros(n_basefuncs_T); dTₑ = zeros(n_basefuncs_T)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_T, cell); Ferrite.reinit!(cellvalues_v, cell)
        celld = celldofs(cell)
        T_celldofs = @view celld[T_range]; v_celldofs = @view celld[v_range]
        Tₑ .= @views u[T_celldofs]; vₑ .= @views u[v_celldofs]
        fill!(dTₑ, 0.0)
        temp_advection_element!(dTₑ, Tₑ, vₑ, cellvalues_T, cellvalues_v)
        assemble!(du, T_celldofs, dTₑ)
    end
    return
end

# ---- Element-level Jacobian blocks ----

# C(v,v): d/dv [-(v·∇v')·φⱼ] = -(φᵢ·∇v' + v·∇φᵢ')·φⱼ
function navierstokes_jac_element!(Jₑ, vₑ, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)
        v  = function_value(cellvalues_v, q_point, vₑ)
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            for i in 1:n_basefuncs
                φᵢ  = shape_value(cellvalues_v, q_point, i)
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                Jₑ[j, i] -= (φᵢ ⋅ ∇v' + v ⋅ ∇φᵢ') ⋅ φⱼ * dΩ
            end
        end
    end
    return
end

# C(T,T): d/dT [-(v·∇T)·φₖ] = -φᵢ (v·∇φⱼ)
function temp_jac_element!(Jₑ, vₑ, cellvalues_T, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_T)
    for q_point in 1:getnquadpoints(cellvalues_T)
        dΩ = getdetJdV(cellvalues_T, q_point)
        v  = function_value(cellvalues_v, q_point, vₑ)
        for i in 1:n_basefuncs
            φᵢ = shape_value(cellvalues_T, q_point, i)
            for j in 1:n_basefuncs
                ∇φⱼ = shape_gradient(cellvalues_T, q_point, j)
                Jₑ[i, j] -= (φᵢ * (v ⋅ ∇φⱼ)) * dΩ
            end
        end
    end
    return
end

# C(T,v): d/dv [-(v·∇T)·φₖ] = -(φᵢ·∇T)·φₖ   (off-diagonal coupling block)
function temp_vel_jac_element!(Jₑ, Tₑ, cellvalues_T, cellvalues_v)
    nT = getnbasefunctions(cellvalues_T)
    nv = getnbasefunctions(cellvalues_v)
    for q_point in 1:getnquadpoints(cellvalues_T)
        dΩ = getdetJdV(cellvalues_T, q_point)
        ∇T = function_gradient(cellvalues_T, q_point, Tₑ)
        for k in 1:nT
            φT = shape_value(cellvalues_T, q_point, k)
            for i in 1:nv
                φv = shape_value(cellvalues_v, q_point, i)
                Jₑ[k, i] -= (φv ⋅ ∇T) * φT * dΩ
            end
        end
    end
    return
end

# ---- Global Jacobian: J = K + C(u) ----
function navierstokes_temp_jac!(J::SparseMatrixCSC, u_uc::Vector, p::RHSparams, t::Float64)
    @unpack K, ch, dh, cellvalues_v, cellvalues_T, u = p
    u .= u_uc; update!(ch, t); apply!(u, ch)

    J .= K   # start from linear part K

    v_range = dof_range(dh, :v)
    T_range = dof_range(dh, :T)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_T = getnbasefunctions(cellvalues_T)

    Jvₑ   = zeros(n_basefuncs_v, n_basefuncs_v)
    JTₑ   = zeros(n_basefuncs_T, n_basefuncs_T)
    JT_vₑ = zeros(n_basefuncs_T, n_basefuncs_v)
    vₑ = zeros(n_basefuncs_v); Tₑ = zeros(n_basefuncs_T)

    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell); Ferrite.reinit!(cellvalues_T, cell)
        celld = celldofs(cell)
        v_celldofs = @view celld[v_range]; T_celldofs = @view celld[T_range]
        vₑ .= @views u[v_celldofs]; Tₑ .= @views u[T_celldofs]
        fill!(Jvₑ, 0.0); fill!(JTₑ, 0.0); fill!(JT_vₑ, 0.0)

        navierstokes_jac_element!(Jvₑ, vₑ, cellvalues_v)
        temp_jac_element!(JTₑ, vₑ, cellvalues_T, cellvalues_v)
        temp_vel_jac_element!(JT_vₑ, Tₑ, cellvalues_T, cellvalues_v)

        # Scatter C(v,v) into global J
        for i_local in 1:length(v_celldofs), j_local in 1:length(v_celldofs)
            J[v_celldofs[i_local], v_celldofs[j_local]] += Jvₑ[i_local, j_local]
        end
        # Scatter C(T,T) into global J
        for i_local in 1:length(T_celldofs), j_local in 1:length(T_celldofs)
            J[T_celldofs[i_local], T_celldofs[j_local]] += JTₑ[i_local, j_local]
        end
        # Scatter C(T,v) off-diagonal block into global J
        for i_local in 1:length(T_celldofs), j_local in 1:length(v_celldofs)
            J[T_celldofs[i_local], v_celldofs[j_local]] += JT_vₑ[i_local, j_local]
        end
    end

    return apply!(J, ch)  # apply Dirichlet constraints to Jacobian
end

println("Nonlinear operators and Jacobian defined.")

---
## 11. ODE Problem Setup & Time Integration <a name="ode"></a>

### The Semi-Discrete System

After FE discretization in space, we obtain a **differential-algebraic system (DAE)**:

$$M \dot{U} = R(U, t), \qquad R(U,t) = KU + N(U)$$

where $M$ is **singular** (pressure block is zero), making this a DAE of **index 2**.

### Solver: Rodas5P

We use the `Rodas5P` algorithm from OrdinaryDiffEq.jl — a **5th-order stiffly accurate Rosenbrock method** suitable for stiff DAEs. Key properties:
- **Implicit**: solves a linear system at each internal stage
- **L-stable**: no order reduction for stiff problems
- **Adaptive**: adjusts time step based on local error estimate
- **`autodiff = false`**: use our analytical Jacobian `navierstokes_temp_jac!`

### Error Norm

A custom `FreeDofErrorNorm` restricts the error norm to **free (unconstrained) DOFs**, so that prescribed Dirichlet DOFs do not artificially inflate the error measure:

$$\text{err} = \left\| u[\text{free\_dofs}] \right\|_\text{rms}$$

### Discontinuity at $t = 1$

The inflow ramp $v_{in}(t) = \min(1.5t, 1.5)$ has a kink at $t = 1$ s. The `d_discontinuities = [1.0]` argument tells the solver to step exactly to $t=1$ and restart, preventing loss of accuracy from integrating through a non-smooth point.

### Time Parameters

| Parameter | Value |
|---|---|
| Simulation time | $t \in [0, 6]$ s |
| Initial step size | $dt = 0.001$ s |
| Absolute tolerance | $10^{-4}$ |
| Relative tolerance | $10^{-5}$ |

In [ ]:
println("Assembling system...")

# --- Define ODE function with mass matrix M and analytical Jacobian ---
rhs = ODEFunction(
    navierstokes_temp!;
    mass_matrix = M,        # M * du/dt = R(u) — singular mass matrix (DAE)
    jac = navierstokes_temp_jac!,   # analytical Jacobian for Newton solver
    jac_prototype = jac_sparsity    # sparsity pattern for efficient allocation
)

# Time span: 0 → 6 seconds
problem = ODEProblem(rhs, u₀, (0.0, 6.0), p)

# --- Custom error norm: only evaluate on free (unconstrained) DOFs ---
struct FreeDofErrorNorm
    ch::ConstraintHandler
end
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

# --- Rodas5P: 5th-order stiffly accurate Rosenbrock method ---
timestepper = Rodas5P(
    autodiff = false,          # use provided analytical Jacobian
    step_limiter! = ferrite_limiter!  # enforce BCs after each accepted step
)

# --- Initialize integrator ---
integrator = init(
    problem, timestepper;
    initializealg = NoInit(),  # skip DAE initialization (BCs set via limiter)
    dt = 0.001,                # initial time step
    adaptive = true,           # enable adaptive time stepping
    abstol = 1.0e-4,           # absolute tolerance
    reltol = 1.0e-5,           # relative tolerance
    progress = true,
    progress_steps = 1,
    verbose = true,
    internalnorm = FreeDofErrorNorm(ch),
    d_discontinuities = [1.0]  # ramp kink at t=1
)

println("Integrator initialized.")

---
## 12. VTK Output & Time Integration Loop <a name="vtk"></a>

### Time Integration Loop

We use `intervals(integrator)` to iterate over solved time intervals. At each step:
1. The integrator advances to the next accepted time $t_n$
2. The solution $U_n = [\mathbf{v}_n, p_n, T_n]$ is extracted
3. Fields are written to a `.vtu` file indexed by step number

### Paraview Collection (`.pvd`)

A `paraview_collection` (`.pvd` file) bundles all per-step `.vtu` files with their timestamps. This allows Paraview to:
- Animate the vortex shedding over time
- Use time-series filters (streamlines, probes)
- Export movies

### Output Fields

`write_solution(vtk, dh, u)` writes **all three fields** to the VTK file:
- `v` — 2D velocity vector field $\mathbf{v}(x,y)$
- `p` — scalar pressure field $p(x,y)$
- `T` — scalar temperature field $T(x,y)$

### Expected Behavior

| Time range | Flow regime |
|---|---|
| $t \in [0, 1]$ | Startup transient; inflow ramping up |
| $t \in [1, 2]$ | Symmetric steady wake forming |
| $t > 2$ | **Vortex shedding onset** — periodic vortex street |
| $t \in [4, 6]$ | Fully periodic oscillation; temperature plumes shed with vortices |

The temperature field follows the velocity closely due to the high Péclet number.

In [ ]:
println("Solving: time integration loop starting...")

# Create Paraview collection file (.pvd)
pvd = paraview_collection("vortex-street-with-temp")

# Iterate over each accepted ODE interval (u, t) = solution at time t
for (step, (u, t)) in enumerate(intervals(integrator))
    println("Step $step  |  t = $(round(t; digits=4)) s  |  max|u| = $(round(maximum(abs, u); sigdigits=4))")

    # Write solution fields (v, p, T) to a .vtu file and register in .pvd
    VTKGridFile("vortex-street-with-temp-$step", dh) do vtk
        write_solution(vtk, dh, u)   # writes :v, :p, :T fields
        pvd[t] = vtk                 # associate with time t in the collection
    end
end

# Save the Paraview collection manifest
vtk_save(pvd)
println("Simulation complete! Open 'vortex-street-with-temp.pvd' in Paraview.")

---
## Summary

This notebook documents a complete **finite element solver** for the 2D incompressible Navier-Stokes equations with a passive temperature scalar. The key design choices were:

| Choice | Reason |
|---|---|
| Q2/Q1 mixed elements | LBB stable velocity-pressure pair |
| Taylor-Hood pair | No spurious pressure modes |
| Singular mass matrix (DAE) | Pressure naturally handled as constraint |
| Rodas5P time stepper | 5th-order, L-stable, ideal for stiff DAEs |
| Analytical Jacobian | Faster and more accurate than finite differences |
| Off-diagonal $C_{Tv}$ block | Captures sensitivity of temperature to velocity perturbations |

### Further Extensions
- **Buoyancy coupling (Boussinesq)**: add $\beta(T - T_0)\mathbf{e}_y$ body force to momentum equation
- **Reactive transport**: add source terms $f(T)$ to the temperature equation
- **3D extension**: replace `RefQuadrilateral` with `RefHexahedron`
- **Mesh refinement**: use adaptive mesh refinement (AMR) near the cylinder wake